In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType, ArrayType, DateType
import sys
import os
from delta import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.utils import AnalysisException
from delta.tables import *
import io
import json
from pyspark.sql.functions import col, year, month, dayofmonth, expr

In [0]:
def create_spark_session():
    return SparkSession \
        .builder \
        .appName("File Streaming Demo") \
        .master("local[3]") \
        .config("spark.databricks.delta.schema.autoMerge.enabled", "true")\
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .enableHiveSupport()\
        .getOrCreate()

In [0]:
def create_deltaTable_insert_update_rows(spark:SparkSession,columns:list, location:str,merge_condition:str,df:DataFrame):
    if (DeltaTable.isDeltaTable(spark, location)):
        print('tabela delta existente')
        deltaTable = DeltaTable.forPath(spark, location)
        deltaTable.alias('tgt') \
            .merge(
                df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        print('tabela delta inexistente')    
        DeltaTable \
            .create(spark) \
            .addColumns(columns) \
            .location(location) \
            .execute()
        deltaTable = DeltaTable.forPath(spark, location)
        deltaTable.alias('tgt') \
            .merge(
                df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()


In [0]:
location_bronze = '/FileStore/bronze/dados_degue/casos_dengue'


#### Leitura dos dados da camada bronze em dataframe spark

In [0]:
location_bronze = '/FileStore/bronze/dados_degue/casos_dengue'
df_bronze = spark.read.format('delta').load(location_bronze)
df_bronze.display()

id,data_iniSE,casos,ibge_code,cidade,uf,cep,latitude,longitude
0,2015-11-08,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
1,2015-12-27,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
2,2019-12-15,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
3,2015-09-20,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
4,2015-12-13,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
5,2015-06-14,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
6,2015-05-10,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
7,2018-09-16,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
8,2018-10-21,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613
9,2016-10-16,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613


#### Transformações da camada Silver

In [0]:
df_silver = (
    df_bronze
    .withColumnRenamed('ibge_code', 'codigo_ibge')
    .withColumnRenamed('data_iniSE', 'data_medicao')
    .withColumnRenamed('casos', 'quantidade_casos')
    .withColumnRenamed('uf', 'estado')
    .withColumn('ano', year('data_medicao'))
    .withColumn('mes', month('data_medicao'))
    .withColumn('dia', dayofmonth('data_medicao'))    
)

In [0]:
df_silver.display()

id,data_medicao,quantidade_casos,codigo_ibge,cidade,estado,cep,latitude,longitude,ano,mes,dia
0,2015-11-08,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2015,11,8
1,2015-12-27,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2015,12,27
2,2019-12-15,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2019,12,15
3,2015-09-20,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2015,9,20
4,2015-12-13,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2015,12,13
5,2015-06-14,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2015,6,14
6,2015-05-10,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2015,5,10
7,2018-09-16,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2018,9,16
8,2018-10-21,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2018,10,21
9,2016-10-16,0.0,230010,Abaiara,CE,63240-000,-7.3364,-39.0613,2016,10,16


In [0]:
path_silver_dengue= '/FileStore/silver/dados_degue/casos_dengue'


#### Processo de Merge/Update para camada Silver

In [0]:
merge_condition = expr("""
    tgt.data_medicao = src.data_medicao 
    and tgt.estado = src.estado 
    and tgt.quantidade_casos = src.quantidade_casos 
    and tgt.codigo_ibge = src.codigo_ibge 
    and tgt.cidade = src.cidade 
    and tgt.cep = src.cep 
    and tgt.latitude = src.latitude 
    and tgt.longitude = src.longitude 
    and tgt.ano = src.ano 
    and tgt.mes = src.`mes`  
    and tgt.dia = src.dia
""")

In [0]:
columns = [
    StructField('id', IntegerType(), True),
    StructField('data_medicao', DateType(), True),
    StructField('quantidade_casos', DoubleType(), True),
    StructField('codigo_ibge', IntegerType(), True),
    StructField('cidade', StringType(), True),
    StructField('estado', StringType(), True),
    StructField('cep', StringType(), True),
    StructField('latitude', DoubleType(), True),
    StructField('longitude', DoubleType(), True),
    StructField('ano', IntegerType(), True),
    StructField('mes', IntegerType(), True),
    StructField('dia', IntegerType(), True)
]

In [0]:
create_deltaTable_insert_update_rows(spark, columns, path_silver_dengue, merge_condition, df_silver)

tabela delta inexistente


In [0]:
#dbutils.fs.rm('/FileStore/silver/dados_degue/casos_dengue', True)